# MAS: Direct LLM vs step 3 vs step 4 (aligned scores)

This notebook scores three **record-level** outputs the same way as `evaluate_3` (**greedy row match** + **per-field** `field_similarity_score`, normalized by GT rows × fields):

1. **Standalone Direct LLM** — `extract_meta_analysis(paper_path, …)` (same family as **Method 1** in `evaluate_3`).
2. **MAS step 3** — workspace artifact **`direct_records`** from player **`direct_extractor`** (intended to mirror the standalone direct call inside the MAS).
3. **MAS step 4** — **`final_meta_analysis_records`** after **`record_extractor`** refinement.

Step 1 (`field_value_pairs`) is only raw evidence for tagging; it is **not** comparable to GT as a table, so this notebook does **not** use step‑1 bag metrics as the main comparison.

**Is MAS step 3 empty?** After running the scoring cell, use **`df_direct.empty`** or **`len(df_direct) == 0`**. If there are no rows, the cell prints a **`peek_workspace_artifact`** dump of `workspace['direct_records']` (Python type, dict keys, or JSON string preview) so you can see whether the artifact is missing, still a string, or valid but with an empty `yield_records` list.

**Why one JSON blob but zero rows before?** Some runs return **one object** whose table fields are **parallel lists** (e.g. eight crops in eight arrays) instead of **`yield_records`: [ eight row-dicts ]**. Pydantic ignores unknown top-level keys, so `yield_records` stayed empty. The shared **`SchemaFactory`** normalizer now **splits** that wide shape into rows before validation. If Jupyter shows **`TextAccessor`**, the artifact was stringified oddly — **`artifact_to_dataframe`** now **`str()`**-coerces and parses JSON when needed.

In [1]:
import json
import re
import sys

sys.path.insert(0, '..')

import pandas as pd

from src.config import LLM_PROVIDER, get_model_name
from src.context import create_context
from src.standards import METADATA_STANDARDS
from src.core.schema_factory import SchemaFactory
from src.experimentutils import (
    load_ground_truth,
    build_study_paper_mapping,
    ProgressOrchestrator,
    evaluate_method_scores,
    print_matching_pairs_from_df,
)
from src.direct_llm_call import extract_meta_analysis

print(f'Provider: {LLM_PROVIDER} | Model: {get_model_name()}')

Provider: google | Model: gemini-3.1-flash-lite-preview


In [2]:
# --- Same study setup as evaluate_3 ---
study_id = 1  # change to probe other papers

gt_df = load_ground_truth()
mapping = build_study_paper_mapping(gt_df)
assert study_id in mapping, f'Study# {study_id} not in mapping'

paper_path = mapping[study_id]
if paper_path.endswith('.pdf'):
    from src.experimentutils import convert_pdf_to_markdown
    paper_path = convert_pdf_to_markdown(paper_path)

standard = METADATA_STANDARDS['wopke_100']
gt_paper = gt_df[gt_df['Study#'] == study_id].reset_index(drop=True)

context = create_context(source=paper_path, name=f'study_{study_id}')

factory = SchemaFactory()
wopke_field_names = list(factory._parse_schema_string(standard).keys())
gt_cols_set = set(gt_paper.columns)
shared_fields = [f for f in wopke_field_names if f in gt_cols_set]

OutputSchema = factory.create_from_standard(
    standard,
    record_class_name='WopkeRecord',
    output_class_name='WopkeOutput',
    records_key='yield_records',
)

print(f'Study# {study_id} | GT rows: {len(gt_paper)} | Shared eval fields: {len(shared_fields)}')

Study# 1 | GT rows: 8 | Shared eval fields: 39


In [3]:
# Standalone direct LLM (same pipeline as evaluate_3 Method 1 — highlighted text inside extract_meta_analysis)
result_standalone_direct = extract_meta_analysis(
    paper_path,
    schema=standard,
    debug_raw_response=False,
)
df_standalone_direct = pd.DataFrame(
    [r.model_dump() for r in result_standalone_direct.yield_records]
)
print(
    f'Standalone direct LLM: {len(df_standalone_direct)} record(s) '
    f'(GT has {len(gt_paper)})'
)

Standalone direct LLM: 8 record(s) (GT has 8)


In [4]:
mas_objective = f'''You are an expert agricultural meta-analysis specialist.
Your task is to extract **intercropping experiment records** from a scientific research paper.
This meta-analysis compares crop yield under intercropping settings versus sole cropping.

**META-ANALYTIC SCHEMA (CRITICAL)**:
{standard}

**SCHEMA RULES**
- Use JSON keys as the EXACT field names in every record.
- Schema descriptions are guidance only — values must be concrete text extracted from the paper.
- Do not rename, add, or remove any schema fields.

**MULTI-RECORD RULE (CRITICAL)**
Each unique combination of crop pair × site × year × treatment level = one SEPARATE record.
Examples:
- 2 density levels × 2 N-treatments = 4 records for the same crop pair.
- Each row in a yield results table is typically a separate record.
Do NOT collapse table rows or merge treatment combinations into a single record.

**YIELD FIELD MAPPING (CRITICAL)**
- `unified yield sc 1` = sole-crop yield of Crop species 1
- `unified yield sc 2` = sole-crop yield of Crop species 2
- `unified yield ic 1` = intercropped yield of Crop species 1
- `unified yield ic 2` = intercropped yield of Crop species 2
Use explicit table headers, row labels, and footnotes to assign yields to the correct species.
If species assignment is ambiguous, set the ambiguous field to null and note the source in `Data source`.
Preserve numeric values exactly — do not round or average.

**SHARED FIELDS**
Fields that are constant across treatments (Year, Lat, Lon, Experimental design, species names,
Intercropping pattern, uniform nutrient inputs) must be filled identically in every record.
Set to null only if genuinely absent from the paper.

**OUTPUT**: One record per unique treatment combination using exact schema field names.
Omit records only if the paper genuinely contains no supporting data.'''

In [5]:
orchestrator = ProgressOrchestrator(topology_name='pipeline')
result_mas = orchestrator.run(
    source=context,
    objective=mas_objective,
    output_schema=OutputSchema,
)
assert result_mas is not None and result_mas.success, 'MAS run failed — check logs/API'

workspace = result_mas.final_workspace
print('Workspace keys:', sorted(workspace.keys()))

Generating plan:   0%|          | 0/1 [00:00<?, ?plan/s]

Executing plan:   0%|          | 0/4 [00:00<?, ?step/s]

  initializing:   0%|          | 0/2 [00:00<?, ?phase/s]

Workspace keys: ['direct_records', 'field_value_pairs', 'final_meta_analysis_records', 'initial_objective', 'labeled_text', 'meta_analytic_schema', 'original_document_text']


In [6]:
def peek_workspace_artifact(name: str, raw) -> None:
    """Print type/shape of a workspace value (why parsing might yield 0 rows)."""
    tname = type(raw).__name__
    print(f'  [{name}] type={tname}', end='')
    if 'Accessor' in tname:
        print('  (often str() → JSON text; notebook coerces before parse)', end='')
    if raw is None:
        print('  value=None')
        return
    if isinstance(raw, dict):
        print(f'  dict_keys={list(raw.keys())[:24]}')
        yr = raw.get('yield_records')
        if yr is not None:
            n = len(yr) if hasattr(yr, '__len__') else None
            print(f'    yield_records: type={type(yr).__name__} len={n}')
    elif isinstance(raw, str):
        print(f'  len={len(raw)}  preview={raw[:280]!r}...')
    else:
        s = str(raw)
        print(f'  preview={s[:280]!r}...')


def artifact_to_dataframe(raw, schema: type) -> pd.DataFrame:
    """Turn workspace artifact into a DataFrame (handles dict, Pydantic, JSON str)."""
    from pydantic import BaseModel

    if raw is None:
        return pd.DataFrame()

    if isinstance(raw, BaseModel):
        if isinstance(raw, schema):
            obj = raw
            recs = getattr(obj, 'yield_records', None) or []
            if not recs:
                print('[artifact_to_dataframe] OK schema but yield_records is empty')
            return pd.DataFrame([r.model_dump() for r in recs])
        parsed = raw.model_dump()
    else:
        parsed = raw

    if isinstance(parsed, str):
        s = parsed.strip()
        s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.IGNORECASE)
        s = re.sub(r'\s*```$', '', s)
        try:
            parsed = json.loads(s)
        except json.JSONDecodeError as e:
            print(f'[artifact_to_dataframe] JSON parse failed: {e}')
            return pd.DataFrame()
    elif not isinstance(parsed, dict):
        # e.g. pandas Styler / TextAccessor: str() often yields the JSON body
        try:
            s = str(parsed).strip()
            s = re.sub(r'^```(?:json)?\s*', '', s, flags=re.IGNORECASE)
            s = re.sub(r'\s*```$', '', s)
            parsed = json.loads(s)
        except (json.JSONDecodeError, TypeError) as e:
            print(f'[artifact_to_dataframe] could not coerce {type(parsed)!r} to JSON dict: {e}')
            return pd.DataFrame()

    try:
        if isinstance(parsed, schema):
            obj = parsed
        elif isinstance(parsed, dict):
            obj = schema.model_validate(parsed)
        else:
            print(f'[artifact_to_dataframe] unexpected type after normalise: {type(parsed)!r}')
            return pd.DataFrame()
        recs = getattr(obj, 'yield_records', None) or []
        if not recs:
            print('[artifact_to_dataframe] OK schema but yield_records is empty')
        return pd.DataFrame([r.model_dump() for r in recs])
    except Exception as e:
        print(f'[artifact_to_dataframe] validate/DataFrame failed: {e}')
        if isinstance(parsed, dict):
            print(f'    top-level keys: {list(parsed.keys())[:30]}')
        return pd.DataFrame()


def aligned_rouge_norm(ext_df: pd.DataFrame, label: str) -> dict:
    if ext_df.empty:
        return {'label': label, 'n_ext': 0, 'n_match': 0, 'norm_rougeL': float('nan')}
    eval_cols = [c for c in shared_fields if c in ext_df.columns]
    if not eval_cols:
        return {'label': label, 'n_ext': len(ext_df), 'n_match': 0, 'norm_rougeL': float('nan')}
    _, overall_norm, _, n_matches = evaluate_method_scores(
        ext_df=ext_df,
        gt_df=gt_paper,
        shared_cols=eval_cols,
        total_records_for_denominator=len(gt_paper),
    )
    return {
        'label': label,
        'n_ext': len(ext_df),
        'n_match': n_matches,
        'norm_rougeL': overall_norm,
    }

### Inspect MAS step 3 (`direct_records`)

Run the cell below after the MAS run and the **helpers** cell (`peek_workspace_artifact`, `artifact_to_dataframe`). It prints the **raw** workspace value and the **parsed** table.

In [7]:
# Raw + parsed view of MAS step 3 (direct_extractor → workspace['direct_records'])
from IPython.display import display

raw_dr = workspace.get('direct_records')
print('=' * 70)
print('STEP 3 — direct_records (raw)')
print('=' * 70)
peek_workspace_artifact('direct_records', raw_dr)

# Long string body (JSON) — increase limit or write to file if needed
if isinstance(raw_dr, str) and len(raw_dr) > 0:
    print('\n--- first 4000 characters of string artifact ---')
    print(raw_dr[:4000])
    if len(raw_dr) > 4000:
        print(f'\n... total length {len(raw_dr)}')

if isinstance(raw_dr, dict):
    import pprint
    print('\n--- dict pretty-print (truncated depth) ---')
    pprint.pp(raw_dr, width=119, depth=4)

print('\n' + '=' * 70)
print('STEP 3 — parsed with OutputSchema → DataFrame')
print('=' * 70)
_df_step3 = artifact_to_dataframe(raw_dr, OutputSchema)
print(f'rows={len(_df_step3)}  columns={len(_df_step3.columns)}')
display(_df_step3)

STEP 3 — direct_records (raw)
  [direct_records] type=TextAccessor  (often str() → JSON text; notebook coerces before parse)  len=1362  preview='{\n    "Year of data": "1980, 1981, 1982, 1984",\n    "Duration of experiment": "4 growing seasons",\n    "Experimental design": "Randomized Complete Block Design",\n    "Sowing date 1": null,\n    "Sowing date 2": null,\n    "Harvest date 1": null,\n    "Harvest date 2": null,\n    "Lat'...

--- first 4000 characters of string artifact ---
{
    "Year of data": "1980, 1981, 1982, 1984",
    "Duration of experiment": "4 growing seasons",
    "Experimental design": "Randomized Complete Block Design",
    "Sowing date 1": null,
    "Sowing date 2": null,
    "Harvest date 1": null,
    "Harvest date 2": null,
    "Lat": null,
    "Lon": null,
    "Crop species 1": "Pisum sativum",
    "Crop species 2": "Hordeum vulgare",
    "Crop type 1": "legume",
    "Crop type 2": "cereal",
    "Intercropping pattern": "Mixed",
    "Density ic 1": 40,
    "D

,Year of data,Duration of experiment,Experimental design,Sowing date 1,Sowing date 2,Harvest date 1,Harvest date 2,Lat,Lon,Crop species 1,...,K input IC1,K input IC2,K total in IC,K Unit,Data source,unified yield sc 1,unified yield sc 2,unified yield ic 1,unified yield ic 2,Yield unit
0,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,456,412,239.5,239.5,g m-2
1,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,357,550,211,330,g m-2
2,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,437,355,234,275,g m-2
3,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,456,467,78,441,g m-2
4,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,714,473,309,349,g m-2
5,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,767,615,200,465,g m-2
6,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,659,332,376,185,g m-2
7,"1980, 1981, 1982, 1984",4 growing seasons,Randomized Complete Block Design,None,None,None,None,None,None,Pisum sativum,...,None,None,None,None,Table 3,661,579,206,401,g m-2


In [8]:
rows = []

r_std = aligned_rouge_norm(df_standalone_direct, 'direct_llm_standalone')
rows.append({
    'stage': r_std['label'] + ' (evaluate_3 Method 1)',
    'n_records': r_std['n_ext'],
    'note': f"matched_rows={r_std['n_match']}/{len(gt_paper)}",
    'aligned_norm_rougeL': r_std['norm_rougeL'],
})

raw_direct = workspace.get('direct_records')
df_direct = artifact_to_dataframe(raw_direct, OutputSchema)
if df_direct.empty:
    print('--- direct_records is empty after parse → workspace peek ---')
    peek_workspace_artifact('direct_records', raw_direct)

r_dir = aligned_rouge_norm(df_direct, 'mas_step3_direct_records')
rows.append({
    'stage': r_dir['label'] + ' (MAS direct_extractor artifact)',
    'n_records': r_dir['n_ext'],
    'note': f"matched_rows={r_dir['n_match']}/{len(gt_paper)}",
    'aligned_norm_rougeL': r_dir['norm_rougeL'],
})

df_final = artifact_to_dataframe(workspace.get('final_meta_analysis_records'), OutputSchema)
r_fin = aligned_rouge_norm(df_final, 'mas_step4_final')
rows.append({
    'stage': r_fin['label'] + ' (MAS record_extractor artifact)',
    'n_records': r_fin['n_ext'],
    'note': f"matched_rows={r_fin['n_match']}/{len(gt_paper)}",
    'aligned_norm_rougeL': r_fin['norm_rougeL'],
})

summary = pd.DataFrame(rows)

def _fmt(x):
    if x is None or (isinstance(x, float) and (x != x)):
        return 'n/a'
    if isinstance(x, float):
        return f'{x:.4f}'
    return str(x)

print('=' * 72)
print(f'  Record-aligned scores  |  Study# {study_id}  |  GT rows: {len(gt_paper)}')
print(f'  Model: {get_model_name()}')
print('=' * 72)
print()
print('  1) Standalone direct LLM (same metric as evaluate_3 Method 1 CSV)')
print(f"    extracted rows      : {r_std['n_ext']}")
print(f"    greedy-matched rows : {r_std['n_match']} / {len(gt_paper)}")
print(f"    aligned norm score  : {_fmt(r_std['norm_rougeL'])}")
print()
print('  2) MAS step 3 — direct_records (direct_extractor)')
print(f"    extracted rows      : {r_dir['n_ext']}")
print(f"    greedy-matched rows : {r_dir['n_match']} / {len(gt_paper)}")
print(f"    aligned norm score  : {_fmt(r_dir['norm_rougeL'])}")
print()
print('  3) MAS step 4 — final_meta_analysis_records (record_extractor)')
print(f"    extracted rows      : {r_fin['n_ext']}")
print(f"    greedy-matched rows : {r_fin['n_match']} / {len(gt_paper)}")
print(f"    aligned norm score  : {_fmt(r_fin['norm_rougeL'])}")
print()
d_sf = r_fin['norm_rougeL'] - r_std['norm_rougeL']
d_fd = r_fin['norm_rougeL'] - r_dir['norm_rougeL']
if d_sf == d_sf:
    print(f"  Delta (step4 − standalone direct): {d_sf:+.4f}")
if d_fd == d_fd:
    print(f"  Delta (step4 − step3 direct_records): {d_fd:+.4f}")
print()
print('-' * 72)
print('  Summary table')
print('-' * 72)
with pd.option_context('display.max_colwidth', 70):
    print(summary.to_string(index=False))
print('=' * 72)

summary

  Record-aligned scores  |  Study# 1  |  GT rows: 8
  Model: gemini-3.1-flash-lite-preview

  1) Standalone direct LLM (same metric as evaluate_3 Method 1 CSV)
    extracted rows      : 8
    greedy-matched rows : 8 / 8
    aligned norm score  : 0.6667

  2) MAS step 3 — direct_records (direct_extractor)
    extracted rows      : 8
    greedy-matched rows : 8 / 8
    aligned norm score  : 0.1629

  3) MAS step 4 — final_meta_analysis_records (record_extractor)
    extracted rows      : 8
    greedy-matched rows : 8 / 8
    aligned norm score  : 0.6437

  Delta (step4 − standalone direct): -0.0230
  Delta (step4 − step3 direct_records): +0.4808

------------------------------------------------------------------------
  Summary table
------------------------------------------------------------------------
                                                   stage  n_records             note  aligned_norm_rougeL
             direct_llm_standalone (evaluate_3 Method 1)          8 matched_row

,stage,n_records,note,aligned_norm_rougeL
0,direct_llm_standalone (evaluate_3 Method 1),8,matched_rows=8/8,0.666667
1,mas_step3_direct_records (MAS direct_extractor...,8,matched_rows=8/8,0.162927
2,mas_step4_final (MAS record_extractor artifact),8,matched_rows=8/8,0.643697


---
## Inspect matching pairs (same as `evaluate_3`)

Greedy row match, then **FIELD / EXTRACTED / GT / SCORE** for:

1. Standalone **direct LLM**  
2. **MAS step 3** (`direct_records`)  
3. **MAS step 4** (`final_meta_analysis_records`)

In [9]:
display_fields = list(shared_fields)

inspect_stages = [
    ('direct_llm_standalone', df_standalone_direct),
    ('mas_step3_direct_records', df_direct),
    ('mas_step4_final_meta_analysis_records', df_final),
]

for method, ext_df in inspect_stages:
    print('=' * 70)
    print(f'  METHOD: {method}')
    print('=' * 70)
    print_matching_pairs_from_df(
        ext_df=ext_df,
        gt_df=gt_paper,
        shared_cols=shared_fields,
        show_fields=display_fields,
    )
    print()

  METHOD: direct_llm_standalone
  Extracted rows : 8
  GT rows        : 8
  Matched pairs  : 8  (8 used crop-label swap)
  Unmatched GT   : 0  (no extracted record assigned)
  Unmatched ext  : 0  (hallucinated / extra records)

  ── Pair 1  (ext row 0  ↔  gt row 0)  mean=0.667  [crop labels swapped]
  FIELD                         EXTRACTED                   GT                          SCORE
  -------------------------------------------------------------------------------------------
  Year of data                  1980                        1980                        1.000
  Duration of experiment        1 growing season            1                           0.500
  Experimental design           Randomized split-plot design  Split-plot                  0.500
  Sowing date 1                 16 April 1980               1980-04-16                  0.000
  Sowing date 2                 16 April 1980               1980-04-16                  0.000
  Harvest date 1                <missin

## Fine-tuning / iteration ideas

- Compare **standalone direct** vs **MAS step 3**: if they diverge, align **direct_extractor** prompts/context with `extract_meta_analysis` (same schema, same highlighting behavior where possible).
- If **step 3 ≈ standalone** but **step 4** is worse, tighten **record_extractor** so refinement does not drop or overwrite good fields.
- If **`direct_records` fails to parse** (`n_records` 0), fix synthesis/schema validation for the `direct_extractor` step before tuning step 4.